## Step 8 — resolve cluster = NULL 
**# of cells in notebook:** 1

**Purpose:** create a `cluster_revised` value for each block by resolving cases where the original cluster field is NULL. For tied blocks where `cluster = NULL` and `tie = 1`, the script uses neighboring blocks to recommend the cluster value based on the value of the block with longest edge. If that is a tie, assign a default value (currently lower density group). Blocks with an existing cluster value keep that value, while `cluster = NULL` & `tie = 0` (no buildings in block) are also assigned a default value (currently lower density group).  

**Input:**

- a geodatabase with the blocks layer from steps 5, 6, and 7
  
**Output:** blocks layer with new fields `cluster_revised`

**Main logic:**

1. Select target blocks where `cluster IS NULL` and `tie = 1`, then select those target blocks plus neighboring/intersecting blocks as candidate features for the neighbor analysis.
2. Run `PolygonNeighbors` on the candidate blocks and, for each target block, sum the shared boundary length by neighboring `cluster` value, ignoring neighbors that also have NULL cluster values.
3. For each target block, recommend the neighboring cluster with the greatest total shared boundary length. If there is a tie for greatest shared boundary length, or if there are no usable clustered edge-neighbors, assign the user-specified default value of 1.
4. Write the final value back to the original blocks layer in `cluster_revised`: preserve the original cluster where it exists; use the neighbor-based recommendation for `cluster = NULL` / `tie = 1`; assign the default value of `1` for `cluster = NULL` / `tie = 0`; and leave unexpected NULL/tie cases as NULL.

In [ ]:
import arcpy
import os
import csv
from collections import defaultdict

# ============================================================
# USER SETTINGS
# ============================================================

blocks_fc = r"E:\World Bank deliverbale 1\_analysis\blocks\blocks.gdb\juba_blocks_20260415_small_utm36n"

out_gdb = r"E:\World Bank deliverbale 1\_analysis\blocks\blocks.gdb"

out_csv = r"E:\World Bank deliverbale 1\_analysis\blocks\cluster_null_tie1_recommendations.csv"

# Existing fields
cluster_field = "cluster"
tie_field = "tie"

# New field to create/update in the original blocks layer
cluster_revised_field = "cluster_revised"

# ------------------------------------------------------------
# User-specified default values
# ------------------------------------------------------------

# Case 1:
# cluster IS NULL and tie = 1, but the neighbor-based outcome is ambiguous.
# Ambiguous means either:
#   - two or more clusters have the same greatest shared boundary length, or
#   - there are no usable clustered edge-neighbors.
AMBIGUOUS_TIE1_CLUSTER_VALUE = "1"

# Case 2:
# cluster IS NULL and tie = 0.
NULL_TIE0_CLUSTER_VALUE = "1"

# ------------------------------------------------------------
# Inspection / QA outputs
# ------------------------------------------------------------

temp_blocks_fc = os.path.join(out_gdb, "_tmp_blocks_cluster_test")
target_blocks_fc = os.path.join(out_gdb, "cluster_null_tie1_target_blocks")
candidate_blocks_fc = os.path.join(out_gdb, "cluster_null_tie1_candidate_blocks")
neighbor_table = os.path.join(out_gdb, "cluster_null_tie1_neighbors")
recommendation_table = os.path.join(out_gdb, "cluster_null_tie1_recommendations")

# Temporary field used to preserve original OBJECTID
orig_oid_field = "orig_oid"

# PolygonNeighbors fields expected when using Report By Fields
src_orig_oid_field = f"src_{orig_oid_field}"
nbr_orig_oid_field = f"nbr_{orig_oid_field}"
src_cluster_field = f"src_{cluster_field}"
nbr_cluster_field = f"nbr_{cluster_field}"
src_tie_field = f"src_{tie_field}"
nbr_tie_field = f"nbr_{tie_field}"
length_field = "LENGTH"

# ============================================================
# ENVIRONMENT
# ============================================================

arcpy.env.overwriteOutput = True

print("============================================================")
print("Cluster recategorization and cluster_revised update")
print("============================================================")
print(f"Input blocks: {blocks_fc}")
print(f"Output GDB:   {out_gdb}")
print(f"Cluster field: {cluster_field}")
print(f"Tie field:     {tie_field}")
print(f"New field:     {cluster_revised_field}")
print("")
print("User-specified default rules:")
print(f"  Ambiguous cluster NULL / tie = 1 result becomes: {AMBIGUOUS_TIE1_CLUSTER_VALUE}")
print(f"  Cluster NULL / tie = 0 becomes:                   {NULL_TIE0_CLUSTER_VALUE}")

desc = arcpy.Describe(blocks_fc)
oid_field = desc.OIDFieldName
spatial_ref = desc.spatialReference
geometry_type = desc.shapeType

print("")
print(f"Input OID field: {oid_field}")
print(f"Geometry type:   {geometry_type}")

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def is_null_like(value):
    """Treat ArcGIS nulls, blanks, and literal '<Null>' as null-like."""
    return (
        value is None
        or str(value).strip() == ""
        or str(value).strip().lower() == "<null>"
    )


def clean_cluster_value(value):
    """Return a normalized cluster value as text, or None if null-like."""
    if is_null_like(value):
        return None
    return str(value).strip()


def parse_tie_value(value):
    """Return tie as int when possible, otherwise None."""
    if value is None:
        return None

    try:
        return int(value)
    except Exception:
        try:
            return int(float(value))
        except Exception:
            return None


def is_tie_value(value, expected):
    parsed = parse_tie_value(value)
    return parsed == expected


def delete_if_exists(path):
    if arcpy.Exists(path):
        print(f"Deleting existing output: {path}")
        arcpy.management.Delete(path)


# ============================================================
# VALIDATE INPUT FIELDS
# ============================================================

existing_fields = {f.name for f in arcpy.ListFields(blocks_fc)}
required_fields = {cluster_field, tie_field}

missing = required_fields - existing_fields
if missing:
    raise ValueError(f"Missing required fields in blocks layer: {missing}")

# ============================================================
# CLEAN OLD INSPECTION OUTPUTS
# ============================================================

for item in [
    temp_blocks_fc,
    target_blocks_fc,
    candidate_blocks_fc,
    neighbor_table,
    recommendation_table
]:
    delete_if_exists(item)

# ============================================================
# STEP 1: CREATE SIMPLIFIED TEMPORARY BLOCKS FEATURE CLASS
#
# This keeps the inspection workflow stable.
# We explicitly preserve the original OBJECTID as orig_oid.
# ============================================================

print("\n============================================================")
print("Step 1: Creating simplified temporary blocks feature class")
print("============================================================")

arcpy.management.CreateFeatureclass(
    out_path=out_gdb,
    out_name=os.path.basename(temp_blocks_fc),
    geometry_type=geometry_type.upper(),
    spatial_reference=spatial_ref
)

arcpy.management.AddField(temp_blocks_fc, orig_oid_field, "LONG")
arcpy.management.AddField(temp_blocks_fc, cluster_field, "TEXT", field_length=100)
arcpy.management.AddField(temp_blocks_fc, tie_field, "LONG")

insert_fields = ["SHAPE@", orig_oid_field, cluster_field, tie_field]
search_fields = ["SHAPE@", oid_field, cluster_field, tie_field]

inserted = 0

with arcpy.da.SearchCursor(blocks_fc, search_fields) as s_cur, \
     arcpy.da.InsertCursor(temp_blocks_fc, insert_fields) as i_cur:

    for shape, oid, cluster_value, tie_value in s_cur:

        cluster_out = clean_cluster_value(cluster_value)
        tie_out = parse_tie_value(tie_value)

        i_cur.insertRow([shape, oid, cluster_out, tie_out])
        inserted += 1

print(f"Temporary blocks created: {temp_blocks_fc}")
print(f"Rows inserted: {inserted:,}")

# ============================================================
# STEP 2: SELECT TARGET BLOCKS
#
# Target blocks are:
# cluster IS NULL AND tie = 1
# ============================================================

print("\n============================================================")
print("Step 2: Selecting target blocks")
print("============================================================")

target_layer = "cluster_test_target_lyr"
candidate_layer = "cluster_test_candidate_lyr"

for lyr in [target_layer, candidate_layer]:
    if arcpy.Exists(lyr):
        arcpy.management.Delete(lyr)

arcpy.management.MakeFeatureLayer(temp_blocks_fc, target_layer)
arcpy.management.MakeFeatureLayer(temp_blocks_fc, candidate_layer)

cluster_delim = arcpy.AddFieldDelimiters(temp_blocks_fc, cluster_field)
tie_delim = arcpy.AddFieldDelimiters(temp_blocks_fc, tie_field)

target_sql = f"{cluster_delim} IS NULL AND {tie_delim} = 1"

print("Selecting target blocks with:")
print(target_sql)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=target_layer,
    selection_type="NEW_SELECTION",
    where_clause=target_sql
)

target_count = int(arcpy.management.GetCount(target_layer)[0])
print(f"Target blocks selected: {target_count:,}")

if target_count == 0:
    raise ValueError("No target blocks found where cluster IS NULL and tie = 1.")

arcpy.management.CopyFeatures(target_layer, target_blocks_fc)
print(f"Saved target blocks: {target_blocks_fc}")

target_oids = set()

with arcpy.da.SearchCursor(target_blocks_fc, [orig_oid_field]) as cursor:
    for row in cursor:
        target_oids.add(row[0])

print(f"Target original OIDs recorded: {len(target_oids):,}")

# ============================================================
# STEP 3: SELECT CANDIDATE BLOCKS
#
# Candidate blocks = target blocks + all blocks touching/intersecting
# target blocks.
# ============================================================

print("\n============================================================")
print("Step 3: Selecting candidate blocks")
print("============================================================")

arcpy.management.SelectLayerByLocation(
    in_layer=candidate_layer,
    overlap_type="INTERSECT",
    select_features=target_layer,
    selection_type="NEW_SELECTION"
)

candidate_count = int(arcpy.management.GetCount(candidate_layer)[0])
print(f"Candidate blocks selected: {candidate_count:,}")

if candidate_count == 0:
    raise ValueError("No candidate blocks selected. Something is wrong with the spatial selection.")

arcpy.management.CopyFeatures(candidate_layer, candidate_blocks_fc)
print(f"Saved candidate blocks: {candidate_blocks_fc}")

# ============================================================
# STEP 4: RUN POLYGON NEIGHBORS
#
# We keep this inspection table.
# We do not include both sides.
# We do not include area overlaps.
# Linear units are implicit from UTM 36N.
# ============================================================

print("\n============================================================")
print("Step 4: Running PolygonNeighbors on candidate blocks")
print("============================================================")

arcpy.analysis.PolygonNeighbors(
    in_features=candidate_blocks_fc,
    out_table=neighbor_table,
    in_fields=[orig_oid_field, cluster_field, tie_field],
    area_overlap="NO_AREA_OVERLAP",
    both_sides="NO_BOTH_SIDES",
    cluster_tolerance=None,
    out_linear_units=""
)

print(f"Created neighbor table: {neighbor_table}")

neighbor_fields = [f.name for f in arcpy.ListFields(neighbor_table)]

print("\nNeighbor table fields:")
for f in neighbor_fields:
    print(f"  {f}")

required_neighbor_fields = {
    src_orig_oid_field,
    nbr_orig_oid_field,
    src_cluster_field,
    nbr_cluster_field,
    src_tie_field,
    nbr_tie_field,
    length_field
}

missing_neighbor_fields = required_neighbor_fields - set(neighbor_fields)

if missing_neighbor_fields:
    raise ValueError(
        "The PolygonNeighbors output does not contain the expected fields: "
        f"{missing_neighbor_fields}. Fields found: {neighbor_fields}"
    )

# ============================================================
# STEP 5: SUM SHARED BOUNDARY LENGTH BY NEIGHBORING CLUSTER
#
# Since both_sides = NO_BOTH_SIDES, the target block can appear
# as either src or nbr, so the code checks both directions.
# ============================================================

print("\n============================================================")
print("Step 5: Summing shared boundary lengths by neighbor cluster")
print("============================================================")

target_summary = defaultdict(lambda: defaultdict(float))
neighbor_detail_rows = []

cursor_fields = [
    src_orig_oid_field,
    nbr_orig_oid_field,
    src_cluster_field,
    nbr_cluster_field,
    src_tie_field,
    nbr_tie_field,
    length_field
]

with arcpy.da.SearchCursor(neighbor_table, cursor_fields) as cursor:

    for (
        src_oid,
        nbr_oid,
        src_cluster,
        nbr_cluster,
        src_tie,
        nbr_tie,
        shared_length
    ) in cursor:

        if shared_length is None:
            continue

        try:
            shared_length = float(shared_length)
        except Exception:
            continue

        # Ignore point-only contacts / zero-length edges.
        if shared_length <= 0:
            continue

        # Case 1: source polygon is the target block
        if src_oid in target_oids:
            target_oid = src_oid
            neighbor_oid = nbr_oid
            neighbor_cluster = nbr_cluster

        # Case 2: neighbor polygon is the target block
        elif nbr_oid in target_oids:
            target_oid = nbr_oid
            neighbor_oid = src_oid
            neighbor_cluster = src_cluster

        # Neither side is one of the null-cluster/tie=1 target blocks
        else:
            continue

        neighbor_cluster_clean = clean_cluster_value(neighbor_cluster)

        # Ignore neighbors that also lack cluster values.
        if neighbor_cluster_clean is None:
            continue

        target_summary[target_oid][neighbor_cluster_clean] += shared_length

        neighbor_detail_rows.append({
            "target_oid": target_oid,
            "neighbor_oid": neighbor_oid,
            "neighbor_cluster": neighbor_cluster_clean,
            "shared_boundary_m": shared_length
        })

print(f"Target blocks with at least one clustered edge-neighbor: {len(target_summary):,}")

# ============================================================
# STEP 6: DETERMINE RECOMMENDED CLUSTER
#
# Rules:
#   - If one cluster has greatest total shared boundary length, use it.
#   - If there is a tie, use AMBIGUOUS_TIE1_CLUSTER_VALUE.
#   - If there are no clustered edge-neighbors, use AMBIGUOUS_TIE1_CLUSTER_VALUE.
# ============================================================

print("\n============================================================")
print("Step 6: Determining recommended cluster values")
print("============================================================")

recommendation_rows = []

for target_oid in sorted(target_oids):

    cluster_lengths = target_summary.get(target_oid, {})

    if not cluster_lengths:
        recommendation_rows.append({
            "block_oid": target_oid,
            "recommended_cluster": AMBIGUOUS_TIE1_CLUSTER_VALUE,
            "max_shared_boundary_m": 0.0,
            "total_clustered_boundary_m": 0.0,
            "num_candidate_clusters": 0,
            "is_tie_for_max": 0,
            "cluster_length_summary": "",
            "status": "NO_CLUSTERED_EDGE_NEIGHBOR_ASSIGNED_DEFAULT"
        })
        continue

    total_clustered_boundary = sum(cluster_lengths.values())
    max_len = max(cluster_lengths.values())

    winning_clusters = [
        cl for cl, val in cluster_lengths.items()
        if abs(val - max_len) < 1e-9
    ]

    winning_clusters_sorted = sorted(winning_clusters)
    is_tie_for_max = 1 if len(winning_clusters_sorted) > 1 else 0

    cluster_length_summary = "; ".join(
        f"{cl}:{cluster_lengths[cl]:.3f}"
        for cl in sorted(cluster_lengths.keys())
    )

    if is_tie_for_max:
        recommended_cluster = AMBIGUOUS_TIE1_CLUSTER_VALUE
        status = "TIE_FOR_MAX_ASSIGNED_DEFAULT"
    else:
        recommended_cluster = winning_clusters_sorted[0]
        status = "OK"

    recommendation_rows.append({
        "block_oid": target_oid,
        "recommended_cluster": recommended_cluster,
        "max_shared_boundary_m": max_len,
        "total_clustered_boundary_m": total_clustered_boundary,
        "num_candidate_clusters": len(cluster_lengths),
        "is_tie_for_max": is_tie_for_max,
        "cluster_length_summary": cluster_length_summary,
        "status": status
    })

print(f"Recommendation rows: {len(recommendation_rows):,}")

# ============================================================
# STEP 7: WRITE RECOMMENDATION TABLE
# ============================================================

print("\n============================================================")
print("Step 7: Writing recommendation table and CSV")
print("============================================================")

delete_if_exists(recommendation_table)

arcpy.management.CreateTable(out_gdb, os.path.basename(recommendation_table))

arcpy.management.AddField(recommendation_table, "block_oid", "LONG")
arcpy.management.AddField(recommendation_table, "recommended_cluster", "TEXT", field_length=100)
arcpy.management.AddField(recommendation_table, "max_shared_boundary_m", "DOUBLE")
arcpy.management.AddField(recommendation_table, "total_clustered_boundary_m", "DOUBLE")
arcpy.management.AddField(recommendation_table, "num_candidate_clusters", "LONG")
arcpy.management.AddField(recommendation_table, "is_tie_for_max", "SHORT")
arcpy.management.AddField(recommendation_table, "cluster_length_summary", "TEXT", field_length=1000)
arcpy.management.AddField(recommendation_table, "status", "TEXT", field_length=100)

insert_fields = [
    "block_oid",
    "recommended_cluster",
    "max_shared_boundary_m",
    "total_clustered_boundary_m",
    "num_candidate_clusters",
    "is_tie_for_max",
    "cluster_length_summary",
    "status"
]

with arcpy.da.InsertCursor(recommendation_table, insert_fields) as cursor:
    for row in recommendation_rows:
        cursor.insertRow([
            row["block_oid"],
            row["recommended_cluster"],
            row["max_shared_boundary_m"],
            row["total_clustered_boundary_m"],
            row["num_candidate_clusters"],
            row["is_tie_for_max"],
            row["cluster_length_summary"],
            row["status"]
        ])

print(f"Created recommendation table: {recommendation_table}")

os.makedirs(os.path.dirname(out_csv), exist_ok=True)

with open(out_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=insert_fields)
    writer.writeheader()
    for row in recommendation_rows:
        writer.writerow(row)

print(f"Wrote CSV: {out_csv}")

# ============================================================
# STEP 8: CREATE / UPDATE cluster_revised ON ORIGINAL BLOCKS
# ============================================================

print("\n============================================================")
print("Step 8: Creating/updating cluster_revised field")
print("============================================================")

# Build lookup from recommendation table rows
recommendation_by_oid = {
    row["block_oid"]: row["recommended_cluster"]
    for row in recommendation_rows
}

# Add cluster_revised field if it does not exist.
# Use TEXT because this field will later be concatenated with other values.
existing_fields = {f.name for f in arcpy.ListFields(blocks_fc)}

if cluster_revised_field not in existing_fields:
    print(f"Adding field: {cluster_revised_field}")
    arcpy.management.AddField(
        in_table=blocks_fc,
        field_name=cluster_revised_field,
        field_type="TEXT",
        field_length=100
    )
else:
    print(f"Field already exists; values will be overwritten: {cluster_revised_field}")

updated_total = 0
kept_original_cluster_count = 0
null_tie1_recommended_count = 0
null_tie1_missing_recommendation_count = 0
null_tie0_default_count = 0
null_other_tie_count = 0

update_fields = [
    oid_field,
    cluster_field,
    tie_field,
    cluster_revised_field
]

with arcpy.da.UpdateCursor(blocks_fc, update_fields) as cursor:

    for oid, cluster_value, tie_value, old_cluster_revised in cursor:

        cluster_clean = clean_cluster_value(cluster_value)
        tie_clean = parse_tie_value(tie_value)

        # Case A: original cluster is populated
        if cluster_clean is not None:
            new_cluster_revised = cluster_clean
            kept_original_cluster_count += 1

        # Case B: cluster is null and tie = 1
        elif tie_clean == 1:

            if oid in recommendation_by_oid:
                new_cluster_revised = recommendation_by_oid[oid]
                null_tie1_recommended_count += 1
            else:
                # This should rarely happen because recommendation_rows should
                # include every cluster-null/tie=1 target block.
                new_cluster_revised = AMBIGUOUS_TIE1_CLUSTER_VALUE
                null_tie1_missing_recommendation_count += 1

        # Case C: cluster is null and tie = 0
        elif tie_clean == 0:
            new_cluster_revised = NULL_TIE0_CLUSTER_VALUE
            null_tie0_default_count += 1

        # Case D: cluster is null and tie is something unexpected/null
        else:
            # Keep this as None so unexpected cases are visible.
            # If desired, you can replace this with another user setting.
            new_cluster_revised = None
            null_other_tie_count += 1

        cursor.updateRow([
            oid,
            cluster_value,
            tie_value,
            new_cluster_revised
        ])

        updated_total += 1

print(f"Rows updated: {updated_total:,}")
print("")
print("cluster_revised assignment counts:")
print(f"  Original cluster preserved:                         {kept_original_cluster_count:,}")
print(f"  cluster NULL / tie = 1 from recommendation table:   {null_tie1_recommended_count:,}")
print(f"  cluster NULL / tie = 1 missing recommendation:      {null_tie1_missing_recommendation_count:,}")
print(f"  cluster NULL / tie = 0 assigned default:            {null_tie0_default_count:,}")
print(f"  cluster NULL / tie other or null left as NULL:       {null_other_tie_count:,}")

# ============================================================
# STEP 9: SUMMARY
# ============================================================

print("\n============================================================")
print("Summary")
print("============================================================")

status_counts = defaultdict(int)

for row in recommendation_rows:
    status_counts[row["status"]] += 1

print("Recommendation status counts:")
for status, count in sorted(status_counts.items()):
    print(f"  {status}: {count:,}")

print("")
print("Interpretation:")
print("  OK = assigned to the cluster with the greatest shared boundary.")
print(f"  TIE_FOR_MAX_ASSIGNED_DEFAULT = tie in shared boundary length; assigned {AMBIGUOUS_TIE1_CLUSTER_VALUE}.")
print(f"  NO_CLUSTERED_EDGE_NEIGHBOR_ASSIGNED_DEFAULT = no usable clustered neighbor; assigned {AMBIGUOUS_TIE1_CLUSTER_VALUE}.")
print(f"  cluster NULL / tie = 0 blocks were assigned {NULL_TIE0_CLUSTER_VALUE} in {cluster_revised_field}.")

print("")
print("Inspection outputs kept:")
print(f"  Temporary blocks:       {temp_blocks_fc}")
print(f"  Target blocks:          {target_blocks_fc}")
print(f"  Candidate blocks:       {candidate_blocks_fc}")
print(f"  Neighbor table:         {neighbor_table}")
print(f"  Recommendation table:   {recommendation_table}")
print(f"  Recommendation CSV:     {out_csv}")

print("")
print(f"Original blocks layer updated with field: {cluster_revised_field}")
print("Done.")